# 03 — Detector de paradas v2: apito + bola + movimento + Smart Tagging

## Por que uma v2

A árvore v1 (notebook 02) decide por frame com uma regra só: "a velocidade dos próximos 5 s fica abaixo de 1,1 m/s". Ela erra
de forma sistemática em pontos bem identificados:

| falha da v1 | evidência | causa |
|---|---|---|
| início do D atrasado | mediana de **12,5 s** entre o apito e o início do D | os jogadores continuam correndo depois do apito |
| escanteio só pela metade | a bola some 14–35 s antes do `start_frame`; a v1 marca só o final | na preparação, os jogadores **trotam até a área** a 1,2–1,8 m/s |
| laterais rápidas perdidas | a bola continua visível e a velocidade não cai | a velocidade é a única evidência |
| parada curta perdida | D1 (5,5 s) do GT | não dá tempo de a velocidade cair |

## A ideia: detectar *paradas*, não frames

Uma bola morta é um **intervalo [início, reinício]**, e cada ponta tem o seu melhor sensor:

* **início**: o **apito** (no áudio), a **bola saindo pela linha** e, como último recurso, o colapso do movimento (a v1);
* **reinício**: a âncora do **Smart Tagging**, o fim do trecho v1 (a velocidade futura volta a subir), a **aceleração coletiva** ou a **retomada** da velocidade depois do apito.

Uma **máquina de estados determinística** (`analise/paradas.py`) liga as duas pontas. A confirmação e o reinício **dependem do tipo**, e é aqui que
entra a ideia de colocar o escanteio na árvore: ele entra *depois* de se saber que a bola saiu pela linha de fundo.

```
VIVA ─(início)─► CANDIDATA ─(confirmação do tipo)─► MORTA ─(reinício)─► VIVA
                     └─(nada confirma: vantagem, apito da torcida)─► VIVA
```

| início | confirma se… | fecha em… |
|---|---|---|
| apito | **freio**: a velocidade cai à metade em 4 s; ou existe um reinício marcado | tag → fim do v1 → **retomada** da velocidade |
| bola na linha de fundo | tag compatível (escanteio ou tiro de meta), v1 em 20 s, ou **convergência para a área** (+3 jogadores) | tag → fim do v1 → aceleração |
| bola na lateral | tag compatível (lateral) ou v1 em 12 s | tag → fim do v1 |
| nenhum evento | (ramo v1, precisão ~0,9) | fim do v1 |

Há ainda uma **guarda de jogo corrido**: se a velocidade média passar de 2,2 m/s por mais de 4 s no meio, a parada é rejeitada. E um apito
dentro da janela de um reinício marcado é apito de **autorização** (saída de bola, cobrança), então não abre parada.

**Regras de honestidade.** Nenhum parâmetro é ajustado olhando o GT público. Os parâmetros vêm do Smart Tagging, do notebook 02 ou de
critérios físicos. A seção 6 varre o limiar de SNR do apito no GT e está **marcada como diagnóstico**. Os rótulos manuais (seção 7)
vão decidir esse limiar.

In [ ]:
%matplotlib inline
import os, sys, json, subprocess, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")

HERE = os.path.abspath("") if os.path.exists("paradas.py") else os.path.join(os.path.abspath(""), "analise")
sys.path.insert(0, HERE)
import rotulos as R, arvore as T, eventos as E, paradas as P, audio_apito as A, rotular_paradas as RP
from features_coletivas import load_all

if not os.path.exists(os.path.join(HERE, "..", "viewer", "data", "apito_tactical.bin")):
    raise SystemExit("rode antes: python analise/audio_apito.py")
X, meta = load_all()
S = E.Sinais(X, meta["index"])
N, FPS = S.N, R.FPS
gt = R.gt_public(); cov = gt != "X"; yD = gt == "D"; g = S.graded
segs = R.gt_dead_segments(gt)
sp = R.set_pieces()
base125, base50 = T.load_baseline("baseline_smooth125"), T.load_baseline("baseline_smooth50")
gt_team = np.where(np.isin(gt, ["H", "A"]), gt, base125)
OUT = os.path.join(HERE, "out", "nb03"); os.makedirs(OUT, exist_ok=True)
sns.set_theme(style="whitegrid")
minute = lambda fr: np.asarray(fr) / FPS / 60
print("{} apitos detectados, f0 = {:.0f} Hz (calibrado nas saídas de bola: {})".format(
    len(S.apito_meta["eventos"]), S.apito_meta["f0_hz"], S.apito_meta.get("f0_por_saida_de_bola_hz")))

## 1. M1: o apito no áudio

O vídeo tático traz o som do estádio no mesmo relógio dos frames. O apito de um árbitro é um **tom quase puro de frequência fixa**
(aqui ~3,8 kHz, com um segundo tom em ~4,3 kHz). A torcida é banda larga, e os cantos e apitos dela mudam de nota.
O detector mede o quanto a banda estreita f0 ± 125 Hz se destaca das bandas vizinhas (SNR em dB).

**Calibração de f0 sem GT.** O árbitro sempre apita a saída de bola. Nas 6 saídas marcadas pelo Smart Tagging, pega-se o tom estreito mais forte e
usa-se a mediana. Tentei antes contar os tons no jogo inteiro, mas a torcida produz mais tons estáveis que o apito.

In [ ]:
def audio_trecho(t0, dur):
    cmd = [A.ffmpeg_exe(), "-v", "error", "-ss", str(t0), "-t", str(dur), "-i", A.find_video(), "-vn", "-ac", "1",
           "-ar", str(A.SR), "-f", "s16le", "-"]
    return np.frombuffer(subprocess.run(cmd, capture_output=True).stdout, np.int16).astype(np.float32) / 32768

f0 = S.apito_meta["f0_hz"]
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (a, b) in zip(axes, segs):
    t0 = a / FPS - 5
    f, t, L = A.spectrogram_db(audio_trecho(t0, 10))
    sel = (f > 1500) & (f < 5500)
    ax.pcolormesh(t - 5, f[sel], L[sel], shading="auto", cmap="magma", vmin=-110, vmax=-45)
    ax.axhline(f0, color="cyan", lw=0.6, ls=":"); ax.axvline(0, color="w", lw=1)
    ax.set_title("início de D (frame {}) ± 5 s".format(a), fontsize=9); ax.set_xlabel("s")
axes[0].set_ylabel("Hz")
fig.suptitle("espectrograma no início dos 4 trechos D do GT (linha ciano = f0 do apito)"); fig.tight_layout(); plt.show()

w0, w1 = np.flatnonzero(cov)[[0, -1]]
fr = np.arange(w0 - 25 * FPS, w1)
fig, ax = plt.subplots(figsize=(16, 3))
ax.plot(minute(fr), S.apito_snr[fr], lw=0.5, color="k")
ax.axhline(10, color="r", ls="--", lw=0.8, label="limiar 10 dB")
for a, b in segs: ax.axvspan(minute(a), minute(b), color="red", alpha=0.15)
for s_, e_, snr in S.apito_meta["eventos"]:
    if fr[0] <= s_ <= fr[-1]: ax.plot(minute(s_), snr, "v", color="orange", ms=8)
ax.set_ylim(-5, 40); ax.set_ylabel("SNR do apito [dB]"); ax.set_xlabel("minuto"); ax.legend()
ax.set_title("SNR por frame na janela do GT — triângulos = apitos detectados, faixas = D"); plt.show()

## 2. M2 e M3: eventos de início e de reinício

* **Saída da bola.** O ATD prende a bola dentro do campo, então a saída aparece de dois jeitos: (a) a bola **some** depois de estar a < 2 m
  de uma linha ou indo para fora, com a velocidade extrapolada por 0,4 s; (b) a bola fica **colada na linha** (|distância| < 0,3 m) por mais de 0,6 s.
  A linha diz o tipo provável: linha de fundo (escanteio ou tiro de meta), lateral, centro.
* **Reinício pelo Smart Tagging.** O `start_frame` não é o reinício. O deslocamento típico de cada tipo é calibrado contra a aceleração coletiva seguinte, sem GT.

In [ ]:
inicio = E.eventos_inicio(S)
offsets, off_tab = E.calibrar_offset_smart_tagging(S)
print("eventos de início:", pd.Series([e["fonte"] + " / " + e["tipo"] for e in inicio]).value_counts().to_dict())
display(off_tab.groupby("tipo").offset_s.agg(["median", "count", lambda v: v.notna().sum()])
        .rename(columns={"median": "deslocamento_mediano_s", "count": "n", "<lambda_0>": "com_aceleracao"}).round(1))

rows = []
for k, (a, b) in enumerate(segs):
    near = [e for e in inicio if -15 * FPS <= e["frame"] - a <= 5 * FPS]
    rows.append(dict(trecho="D{}".format(k + 1), inicio_GT=a,
                     eventos=", ".join("{} {} ({:+.1f} s)".format(e["fonte"], e["tipo"], (e["frame"] - a) / FPS) for e in near)))
display(pd.DataFrame(rows))

cob = []
for t in sp.type.unique():
    st = sp.start_frame[sp.type == t].values
    hit = [any(s - 45 * FPS <= e["frame"] <= s for e in inicio) for s in st]
    cob.append(dict(tipo=t, set_pieces=len(st), com_evento_de_inicio=int(np.sum(hit))))
display(pd.DataFrame(cob))

## 3. M4: a máquina de estados no GT

In [ ]:
cfg = P.Config()
dead_v2, tab = P.detectar_paradas(S, cfg, offsets=offsets)
dead_v1, _ = P.v1_runs(S, cfg)
display(tab[(tab.inicio > w0 - 30 * FPS) & (tab.inicio < w1)].round(2))
print("início/fim de cada trecho D do GT (s; negativo = antes):")
rep = pd.concat({"v1": T.dead_segment_report(dead_v1, gt), "v2": T.dead_segment_report(dead_v2, gt)}, axis=1)
display(rep.round(2))

In [ ]:
COLS = {"H": "#1f6fd1", "A": "#ee2523", "D": "#111111"}
TCOL = {"apito": "#c0ca33", "linha_de_fundo": "#ab47bc", "lateral": "#26a69a", "centro": "#ff7043",
        "campo": "#8d6e63", "desconhecida": "#78909c", "sem_evento": "#546e7a"}
fr = np.arange(w0, w1 + 1)
fig, axes = plt.subplots(5, 1, figsize=(16, 5.6), sharex=True, gridspec_kw=dict(height_ratios=[1, 1, 1, 1, 2]))
for lab, c in COLS.items():
    for s_, e_ in R.runs(gt[fr] == lab): axes[0].axvspan(minute(fr[s_]), minute(fr[min(e_, len(fr) - 1)]), color=c, lw=0)
for s_, e_ in R.runs(dead_v1[fr]): axes[1].axvspan(minute(fr[s_]), minute(fr[min(e_, len(fr) - 1)]), color="k", lw=0)
for r in tab.itertuples():
    if r.fim > w0 and r.inicio < w1:
        axes[2].axvspan(minute(max(r.inicio, w0)), minute(min(r.fim, w1)), color=TCOL.get(r.tipo, "grey"), lw=0)
for s_, e_, snr in S.apito_meta["eventos"]:
    if w0 <= s_ <= w1: axes[3].axvline(minute(s_), color="orange", lw=2)
for e in inicio:
    if w0 <= e["frame"] <= w1 and e["fonte"] == "saida_bola": axes[3].axvline(minute(e["frame"]), color="purple", lw=2, ls="--")
axes[4].plot(minute(fr), S.speed1[fr], color="k", lw=0.8); axes[4].set_ylabel("vel. 1 s")
for s_, e_ in segs: axes[4].axvspan(minute(s_), minute(e_), color="red", alpha=0.12)
for ax, lab in zip(axes[:4], ["GT", "v1", "v2 (cor = tipo)", "apito / bola na linha"]):
    ax.set_yticks([]); ax.set_ylabel(lab, rotation=0, ha="right")
from matplotlib.patches import Patch
axes[2].legend(handles=[Patch(color=c, label=k) for k, c in TCOL.items()], ncol=7, fontsize=7, loc="upper right")
axes[-1].set_xlabel("minuto"); fig.tight_layout(); fig.savefig(os.path.join(OUT, "linha_do_tempo_v2.png"), dpi=110); plt.show()

## 4. Ablação: o que cada método acrescenta

Cada linha liga um componente a mais. As colunas:

* `precisao_D`, `recall_D`, `macroF1`: medidos por frame no GT;
* `spellF1_125`: spellF1 com os H/A do `baseline_smooth125`;
* `spellF1_timeGT`: spellF1 com os times do GT, que isola o efeito do D;
* `cob_*`: fração da janela de cada tipo de set piece marcada como D, no jogo inteiro, sem GT. A janela vai do `start_frame` menos uma
  antecedência típica do tipo (escanteio 20 s, tiro de meta 15 s, lateral e falta 8 s, saída de bola 20 s) até o reinício.

In [ ]:
COVWIN = {"Corner Kick": 20, "Goal Kick": 15, "Throw-in": 8, "Free Kick": 8, "Kick Off (After Goal)": 20}
def cobertura(d):
    return {"cob_" + t.split()[0].lower(): np.mean([d[s - w * FPS:s + int(offsets.get(t, 3) * FPS)].mean()
                                                    for s in sp.start_frame[sp.type == t]]) for t, w in COVWIN.items()}

def linha(nome, d):
    tp = (d & yD)[cov].sum(); fp = (d & ~yD)[cov].sum(); fn = (~d & yD)[cov].sum()
    a = T.evaluate(gt, T.combine_with_baseline(base125, d, g)); b = T.evaluate(gt, T.combine_with_baseline(gt_team, d, g))
    rep = T.dead_segment_report(d, gt)
    return dict(variante=nome, precisao_D=tp / max(1, tp + fp), recall_D=tp / max(1, tp + fn), macroF1=a["macro_f1"],
                spellF1_125=a["spell_f1_mean"], spellF1_timeGT=b["spell_f1_mean"],
                erro_inicio_mediano_s=np.nanmedian(np.abs(rep.atraso_inicio_s)), D_no_jogo=d[g].mean(),
                paradas=len(T.runs(d & g)), **cobertura(d))

VAR = [("v1 (notebook 02)", None),
       ("+ apito", P.Config(usar_inicio=("apito",))),
       ("+ bola na linha", P.Config(usar_inicio=("saida_bola",))),
       ("completo, sem ramos por tipo", P.Config(ramos_por_tipo=False)),
       ("completo (v2)", P.Config())]
abl = []
for nome, c in VAR:
    d = dead_v1 if c is None else P.detectar_paradas(S, c, offsets=offsets)[0]
    abl.append(linha(nome, d))
abl = pd.DataFrame(abl)
abl.to_csv(os.path.join(OUT, "ablacao.csv"), index=False)
display(abl.style.format({c: "{:.3f}" for c in abl.columns if abl[c].dtype.kind == "f"})
        .background_gradient(subset=["recall_D", "macroF1", "spellF1_timeGT"] + [c for c in abl.columns if c.startswith("cob_")], cmap="Greens")
        .background_gradient(subset=["erro_inicio_mediano_s"], cmap="Reds"))

In [ ]:
cc = abl.set_index("variante")[[c for c in abl.columns if c.startswith("cob_")]].loc[["v1 (notebook 02)", "completo (v2)"]].T
ax = cc.plot.barh(figsize=(8, 3.5), color=["#999999", "#2ca02c"]); ax.set_xlim(0, 1)
ax.set_title("fração da parada marcada como D, por tipo de reinício (jogo inteiro, sem GT)"); plt.show()

**Leitura** (números da execução abaixo; com 4 trechos no GT, diferenças de ±0,02 são ruído):

* **O apito resolve o início.** No GT, o erro do início do D cai de 5–9 s (v1) para ~1 s, e o trecho curto D1, que a v1 perdia, aparece.
* **A bola na linha resolve o escanteio e o tiro de meta.** A cobertura dos escanteios sobe de ~0,76 para ~0,93, porque a parada começa quando
  a bola sai e não quando os jogadores param.
* **Os ramos por tipo aumentam a precisão.** Sem eles, um sumiço de bola na lateral seguido de uma falta mais tarde vira uma parada longa
  que cobre jogo vivo. Com eles, o tipo do início precisa ser compatível com o reinício.
* **A precisão não foi sacrificada.** O recall do D sobe de 0,63 para 0,99 e a precisão fica em ~0,89. O ganho no spellF1 com o
  `baseline_smooth125` é pequeno (0,471 → 0,485) porque ali quem limita são os rótulos H/A do baseline. Com os times do GT, que mostram só o efeito do D,
  o spellF1 vai de 0,895 a 1,0.
* **Os componentes se complementam.** Só com o apito, a cobertura dos escanteios não melhora. Só com a bola na linha, as faltas e o D1 continuam perdidos.
  O detector completo é o único com erro de início < 1 s **e** cobertura alta.

## 5. Por que o limiar do apito está em 10 dB

Sem GT, o critério é a fração de apitos que acabam virando parada, por faixa de SNR. Se apitos fracos fossem quase sempre ruído,
essa fração cairia nas faixas baixas.

In [ ]:
_, tab0 = P.detectar_paradas(S, P.Config(apito_min_snr=0), offsets=offsets)
starts = tab0[tab0.fonte_inicio == "apito"].inicio.values
ap = E.eventos_apito(S, min_snr=0)
faixa = []
for lo, hi in [(10, 13), (13, 16), (16, 20), (20, 99)]:
    sel = [e for e in ap if lo <= e["forca"] < hi]
    faixa.append(dict(snr_dB="{}–{}".format(lo, hi), apitos=len(sel),
                      viraram_parada=np.mean([np.any(np.abs(starts - e["frame"]) <= FPS) for e in sel]) if sel else np.nan,
                      v1_dispara_em_20s=np.mean([dead_v1[e["frame"]:e["frame"] + 20 * FPS].any() for e in sel]) if sel else np.nan))
display(pd.DataFrame(faixa).round(2))

A fração **não cai** nas faixas baixas, então os dados sem GT não justificam um limiar mais alto, e o padrão fica em 10 dB.

## 6. Diagnóstico otimista: o limiar do apito visto pelo GT

**Não use esta seção para escolher.** Ela varre o limiar olhando o GT. Com o f0 calibrado nas saídas de bola, o resultado no GT
quase não depende do limiar (10–20 dB): os apitos que importam são fortes. Fora do GT, o limiar muda o número de paradas, e é isso que os
rótulos manuais vão calibrar.

Nota de desenvolvimento: numa versão anterior, com f0 = 3789 Hz e uma banda vizinha mais larga, havia um apito falso a 497 s. Ele sumiu
com a calibração atual (3844 Hz), que **não usa o GT**.

In [ ]:
diag = [linha("SNR ≥ {}".format(t), P.detectar_paradas(S, P.Config(apito_min_snr=t), offsets=offsets)[0]) for t in (10, 13, 16, 20)]
display(pd.DataFrame(diag)[["variante", "precisao_D", "recall_D", "macroF1", "spellF1_125", "spellF1_timeGT", "paradas"]].round(3))

## 7. Rótulos manuais

`python analise/rotular_paradas.py` gera `analise/out/rotulos_manuais.csv`, com ~44 paradas embaralhadas e separadas por categoria
(escanteio, lateral, tiro de meta, apito que abriu parada, apito descartado, parada só por velocidade, controle de jogo vivo).
Para cada linha: abra o viewer em `frame_ref` (ou use **N** / **Shift+N**), anote em `inicio_real` o frame em que a bola deixou de estar em jogo e
em `fim_real` o frame do reinício. Se ali não houve parada, escreva `viva` em `obs`.

Assim que houver linhas preenchidas, a célula abaixo mede a v1 e a v2 nelas.

In [ ]:
if os.path.exists(RP.OUT):
    lab = RP.carregar()
    print("{} linhas rotuladas em {}".format(len(lab), RP.OUT))
    if len(lab):
        res = pd.concat({"v1": RP.avaliar(lab, dead_v1).set_index("id"), "v2": RP.avaliar(lab, dead_v2).set_index("id")}, axis=1)
        display(res.round(2))
        for k in ("v1", "v2"):
            r = res[k]
            print(k, "| cobertura média das paradas: {:.2f} | |erro início| mediano: {:.1f} s | falso D nos trechos viva: {:.2f}".format(
                r.get("cobertura", pd.Series(dtype=float)).mean(), r.get("erro_inicio_s", pd.Series(dtype=float)).abs().median(),
                r.get("falso_D_frac", pd.Series(dtype=float)).mean()))
else:
    print("ainda sem rótulos: rode python analise/rotular_paradas.py e preencha no viewer")

## 8. Exportar

| arquivo | conteúdo |
|---|---|
| `evaluation/results/paradas_smooth125.csv` e `paradas_smooth50.csv` | submissões completas (H/A do baseline + D da v2), para o viewer e para o `run_evaluation.py` |
| `analise/out/paradas.csv` | a tabela de paradas (início, fim, tipo, fonte do início e do fim, confirmação) |
| `analise/out/paradas_D.csv` | `frame, morta` |
| `viewer/data/eventos.json` | a linha "paradas e apitos" da timeline do viewer |

In [ ]:
import possession_metrics as pm
for nome, base in (("paradas_smooth125", base125), ("paradas_smooth50", base50)):
    lab = T.combine_with_baseline(base, dead_v2, g)
    path = os.path.join(HERE, "..", "evaluation", "results", nome + ".csv")
    pd.DataFrame({"frame": np.arange(N), "label": lab}).to_csv(path, index=False)
    err = pm.validate_submission(pm.read_label_csv(path), N)
    r = T.evaluate(gt, lab)
    print("{:<20} {} | GT público: spellF1 {:.3f} macroF1 {:.3f}".format(nome, "válida" if err is None else err, r["spell_f1_mean"], r["macro_f1"]))
tab.to_csv(os.path.join(HERE, "out", "paradas.csv"), index=False)
pd.DataFrame({"frame": np.arange(N), "morta": dead_v2.astype(int)}).to_csv(os.path.join(HERE, "out", "paradas_D.csv"), index=False)
print("viewer:", P.exportar_viewer(tab, S))
print("{} paradas no jogo; D em {:.1%} dos frames avaliados".format(len(tab), dead_v2[g].mean()))
display(tab.groupby(["tipo", "fonte_fim"]).size().unstack(fill_value=0))

## 9. Conclusões

1. **Cada ponta da bola morta tem o seu sensor.** O início vem do **apito** (faltas, impedimentos, gols) e da **bola saindo pela linha** (laterais,
   escanteios, tiros de meta, que não são apitados). O reinício vem do Smart Tagging e da física do movimento coletivo.
2. **O escanteio entrou na árvore, no lugar certo.** O ramo "linha de fundo" é ativado pela saída da bola e confirmado pela tag ou pela
   **convergência dos jogadores para a área**, não pela lentidão. É isso que leva a cobertura dos escanteios a ~0,93.
3. **O apito precisa de confirmação.** Um apito sem freio coletivo nem reinício marcado é descartado (vantagem, torcida). Um apito colado a um reinício
   marcado autoriza a cobrança e não abre parada.
4. **O que falta:**
   * os **rótulos manuais**, para calibrar o limiar do apito, os offsets por tipo e a guarda de jogo vivo;
   * a bola do `tracking_bola`, para a saída pela linha, onde o ATD perde a bola;
   * um classificador de **posse** melhor, porque com os times do baseline o spellF1 fica limitado pelo H/A.